In [1]:

from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:

%pip install -q pandas pyarrow duckdb


In [3]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/content/drive/MyDrive/RAG UFC-AI-Analyst")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
CLEAN_CSV_PATH = PROCESSED_DIR / "ufc_fights_clean.csv"
DB_PATH = DATA_DIR / "ufc.duckdb"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
if not CLEAN_CSV_PATH.is_file():
    raise FileNotFoundError(f"Cleaned CSV not found: {CLEAN_CSV_PATH}. Complete Notebook 01 first.")

df = pd.read_csv(CLEAN_CSV_PATH, low_memory=False)
print("Input CSV:", CLEAN_CSV_PATH)
print("Dataset shape:", df.shape)
display(df["fight_outcome"].value_counts(dropna=False).to_frame("count"))


Input CSV: /content/drive/MyDrive/RAG UFC-AI-Analyst/data/processed/ufc_fights_clean.csv
Dataset shape: (8892, 284)


,count
fight_outcome,
red_win,5610
blue_win,3281
draw,1


In [4]:

required = [
    "fight_id", "event_id", "event_name", "event_date", "event_location",
    "fighter_a_id", "fighter_a_name", "fighter_b_id", "fighter_b_name",
    "fight_outcome", "weight_class", "method"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if df["fight_id"].isna().any():
    raise ValueError("fight_id contains missing values.")
if df["fight_id"].duplicated().any():
    raise ValueError("fight_id is not unique; inspect source before proceeding.")
print("Schema validation passed.")
print("Rows:", len(df))
print("Unique fight IDs:", df["fight_id"].nunique())


Schema validation passed.
Rows: 8892
Unique fight IDs: 8892


In [5]:

event_columns = ["event_id", "event_name", "event_date", "event_location"]
events = df[event_columns].drop_duplicates(subset=["event_id"]).copy()
events["event_date"] = pd.to_datetime(events["event_date"], errors="coerce")

event_conflicts = (
    df.groupby("event_id", dropna=False)[["event_name", "event_date", "event_location"]]
      .nunique(dropna=False)
)
conflicting_events = event_conflicts[(event_conflicts > 1).any(axis=1)]
print("Events:", events.shape)
print("Event IDs with conflicting attributes:", len(conflicting_events))
display(events.head())


Events: (794, 4)
Event IDs with conflicting attributes: 0


,event_id,event_name,event_date,event_location
0,ufc-2-no-way-out,UFC 2: No Way Out,1994-03-11,"Denver, Colorado, USA"
15,ufc-3-the-american-dream,UFC 3: The American Dream,1994-09-09,"Charlotte, North Carolina, USA"
21,ufc-4-revenge-of-the-warriors,UFC 4: Revenge of the Warriors,1994-12-16,"Tulsa, Oklahoma, USA"
29,ufc-5-the-return-of-the-beast,UFC 5: The Return of the Beast,1995-04-07,"Charlotte, North Carolina, USA"
38,ufc-6-clash-of-the-titans,UFC 6: Clash of the Titans,1995-07-14,"Casper, Wyoming, USA"


In [6]:

fighter_fields = ["id", "name", "dob", "height", "reach"]

fighters_a = df[[
    "fighter_a_id", "fighter_a_name", "fighter_a_dob",
    "fighter_a_height", "fighter_a_reach"
]].copy()
fighters_a.columns = fighter_fields

fighters_b = df[[
    "fighter_b_id", "fighter_b_name", "fighter_b_dob",
    "fighter_b_height", "fighter_b_reach"
]].copy()
fighters_b.columns = fighter_fields

fighters_all = pd.concat([fighters_a, fighters_b], ignore_index=True)
fighters_all = fighters_all.dropna(subset=["id"]).copy()

fighter_conflicts = (
    fighters_all.groupby("id")[["name", "dob", "height", "reach"]]
    .nunique(dropna=True)
)
conflicting_fighter_ids = fighter_conflicts[(fighter_conflicts > 1).any(axis=1)]
print("Fighter IDs with conflicting attributes:", len(conflicting_fighter_ids))

def first_non_null(series):
    values = series.dropna()
    return values.iloc[0] if not values.empty else pd.NA

fighters = (
    fighters_all.groupby("id", as_index=False, sort=False)
    .agg({"name": first_non_null, "dob": first_non_null,
          "height": first_non_null, "reach": first_non_null})
    .rename(columns={"id": "fighter_id"})
)
fighters["dob"] = pd.to_datetime(fighters["dob"], errors="coerce")
print("Fighters:", fighters.shape)
display(fighters.head())


Fighter IDs with conflicting attributes: 582


/tmp/ipykernel_829/2058831003.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fighters["dob"] = pd.to_datetime(fighters["dob"], errors="coerce")


Fighters: (2747, 5)


,fighter_id,name,dob,height,reach
0,429e7d3725852ce9,ROYCE GRACIE,1966-12-12,73.0,<NA>
1,749685d24e2cac50,REMCO PARDOEL,1969-05-23,75.0,<NA>
2,a6c2f5381d575920,ORLANDO WIET,1965-10-24,70.0,<NA>
3,be9d259be012e8a4,SCOTT MORRIS,NaT,70.0,<NA>
4,46c8ec317aff28ac,PATRICK SMITH,1963-08-28,74.0,<NA>


In [7]:

fight_columns = [
    "fight_id", "event_id", "weight_class", "bout_type", "method",
    "round", "time", "time_format", "referee", "bonus", "details"
]
fight_columns = [c for c in fight_columns if c in df.columns]
fights = df[fight_columns].drop_duplicates(subset=["fight_id"]).copy()
print("Fights:", fights.shape)
display(fights.head())


Fights: (8892, 11)


,fight_id,event_id,weight_class,bout_type,method,round,time,time_format,referee,bonus,details
0,00835554f95fa911,ufc-2-no-way-out,Open Weight,UFC 2 Tournament Title Bout,KO/TKO,1.0,1:17,No Time Limit,John McCarthy,belt,Punches to Head From Mount Submission to Strikes
1,17ee4caf06981b81,ufc-2-no-way-out,Open Weight,Open Weight Bout,Submission,1.0,9:51,No Time Limit,John McCarthy,-,Ezekiel Choke From Side Control
2,3b020d4914b44fc8,ufc-2-no-way-out,Open Weight,Open Weight Bout,KO/TKO,1.0,2:50,No Time Limit,John McCarthy,-,to Corner Stoppage
3,4acab67848e78327,ufc-2-no-way-out,Open Weight,Open Weight Bout,Submission,1.0,0:20,No Time Limit,John McCarthy,-,Guillotine Choke From Mount
4,4b9ae533ccb3fcdf,ufc-2-no-way-out,Open Weight,Open Weight Bout,Submission,1.0,0:58,No Time Limit,John McCarthy,-,Guillotine Choke Standing


In [8]:

def make_side(data, side):
    opponent = "b" if side == "a" else "a"
    prefix = f"fighter_{side}_"
    rename_map = {
        f"{prefix}id": "fighter_id",
        f"fighter_{opponent}_id": "opponent_id",
    }
    excluded = {"id", "name", "dob", "height", "reach"}
    for col in data.columns:
        if col.startswith(prefix):
            suffix = col[len(prefix):]
            if suffix not in excluded:
                rename_map[col] = suffix

    selected = {src: dst for src, dst in rename_map.items() if src in data.columns}
    result = data[["fight_id"] + list(selected.keys())].rename(columns=selected).copy()
    outcome = data["fight_outcome"].astype("string").str.strip().str.lower()
    result["is_winner"] = pd.Series(pd.NA, index=result.index, dtype="boolean")

    if side == "a":
        result.loc[outcome.eq("red_win"), "is_winner"] = True
        result.loc[outcome.eq("blue_win"), "is_winner"] = False
    else:
        result.loc[outcome.eq("blue_win"), "is_winner"] = True
        result.loc[outcome.eq("red_win"), "is_winner"] = False
    return result

performances_a = make_side(df, "a")
performances_b = make_side(df, "b")
fight_performances = pd.concat([performances_a, performances_b], ignore_index=True)

print("Fight performances:", fight_performances.shape)
display(fight_performances["is_winner"].value_counts(dropna=False).to_frame("count"))
print("Rows per fight:", fight_performances.groupby("fight_id").size().value_counts().to_dict())


Fight performances: (17784, 129)


,count
is_winner,
True,8891
False,8891
<NA>,2


Rows per fight: {2: 8892}


In [9]:

assert len(fights) == df["fight_id"].nunique()
assert len(fight_performances) == 2 * len(df)
assert fight_performances["fight_id"].nunique() == len(df)

winner_flags_per_fight = fight_performances.groupby("fight_id")["is_winner"].sum(min_count=1)
print("Fights with exactly one winner:", int((winner_flags_per_fight == 1).sum()))
print("Fights with no known winner:", int(winner_flags_per_fight.isna().sum() + (winner_flags_per_fight == 0).sum()))
print("Fights with multiple winners:", int((winner_flags_per_fight > 1).sum()))

unknown_outcomes = df.loc[
    ~df["fight_outcome"].astype("string").str.strip().str.lower().isin(["red_win", "blue_win"]),
    ["fight_id", "fight_outcome", "fighter_a_name", "fighter_b_name"]
]
print("Draw/other/unknown outcome rows:", len(unknown_outcomes))
display(unknown_outcomes.head(10))


Fights with exactly one winner: 8891
Fights with no known winner: 1
Fights with multiple winners: 0
Draw/other/unknown outcome rows: 1


,fight_id,fight_outcome,fighter_a_name,fighter_b_name
8875,e556bdf83f767d4b,draw,Ernesta Kareckaite,Melissa Gatto


In [10]:

tables = {
    "events": events,
    "fighters": fighters,
    "fights": fights,
    "fight_performances": fight_performances,
}
for name, table in tables.items():
    path = PROCESSED_DIR / f"{name}.parquet"
    table.to_parquet(path, index=False)
    print(f"{name}: {table.shape} -> {path}")


events: (794, 4) -> /content/drive/MyDrive/RAG UFC-AI-Analyst/data/processed/events.parquet
fighters: (2747, 5) -> /content/drive/MyDrive/RAG UFC-AI-Analyst/data/processed/fighters.parquet
fights: (8892, 11) -> /content/drive/MyDrive/RAG UFC-AI-Analyst/data/processed/fights.parquet
fight_performances: (17784, 129) -> /content/drive/MyDrive/RAG UFC-AI-Analyst/data/processed/fight_performances.parquet


In [11]:

import duckdb

conn = duckdb.connect(str(DB_PATH), read_only=False)
for name in tables:
    parquet_path = (PROCESSED_DIR / f"{name}.parquet").as_posix().replace("'", "''")
    conn.execute(
        f'CREATE OR REPLACE TABLE "{name}" AS '
        f"SELECT * FROM read_parquet('{parquet_path}')"
    )

print("Database:", DB_PATH)
display(conn.execute("SHOW TABLES").fetchdf())


Database: /content/drive/MyDrive/RAG UFC-AI-Analyst/data/ufc.duckdb


,name
0,events
1,fight_performances
2,fighters
3,fights


In [12]:

for name in ["events", "fighters", "fights", "fight_performances"]:
    count = conn.execute(f'SELECT COUNT(*) FROM "{name}"').fetchone()[0]
    print(f"{name}: {count:,} rows")

print("\nSample fight performances:")
display(conn.execute("SELECT * FROM fight_performances LIMIT 5").fetchdf())


events: 794 rows
fighters: 2,747 rows
fights: 8,892 rows
fight_performances: 17,784 rows

Sample fight performances:


,fight_id,fighter_id,opponent_id,age,weight,sig_strikes_landed,sig_strikes_attempted,total_strikes_landed,total_strikes_attempted,body_strikes_landed,...,round5_leg_strikes_landed,round5_leg_strikes_attempted,round5_distance_strikes_landed,round5_distance_strikes_attempted,round5_clinch_strikes_landed,round5_clinch_strikes_attempted,round5_ground_strikes_landed,round5_ground_strikes_attempted,is_winner,fight_minutes
0,00835554f95fa911,429e7d3725852ce9,46c8ec317aff28ac,27.2,175.0,4.0,4.0,11.0,11.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN
1,17ee4caf06981b81,749685d24e2cac50,6e02c318762f52f1,24.8,260.0,4.0,6.0,20.0,22.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN
2,3b020d4914b44fc8,a6c2f5381d575920,96087e90d900f0ef,28.4,170.0,8.0,12.0,11.0,15.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN
3,4acab67848e78327,be9d259be012e8a4,a683f9ddb70aa4bd,NaN,210.0,1.0,1.0,2.0,2.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN
4,4b9ae533ccb3fcdf,46c8ec317aff28ac,ea0ad155451ed1f5,30.5,225.0,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN


In [13]:

conn.close()
print("Notebook 02 completed. Parquet tables and DuckDB database are saved in Google Drive.")


Notebook 02 completed. Parquet tables and DuckDB database are saved in Google Drive.
